In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata, scvelo as scv

# modern rpy2 imports (3.5+)
from rpy2.robjects import r, globalenv, default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import numpy2ri
from rpy2.robjects import pandas2ri
from rpy2.robjects.vectors import StrVector


# -------------------------------------------------
# Dummy continuous color helper
# -------------------------------------------------
def make_dummy_continuous(color_vec):
    color_vec = np.asarray(color_vec)
    uniq = np.unique(color_vec)
    mapping = {u: i for i, u in enumerate(uniq)}
    return np.array([mapping[x] for x in color_vec], dtype=float)


# -------------------------------------------------
# DATASET REGISTRY
# -------------------------------------------------
DATASETS = {
    "cell_cycle": {
        "X": "./data/real_data_benchmark/cell_cycle/X_cc.npy",
        "V": "./data/real_data_benchmark/cell_cycle/V_cc.npy",
        "color": "./data/real_data_benchmark/cell_cycle/color_cell_cycle_relativePos.npy",
    },
    "pancreas": {
        "X": "./data/real_data_benchmark/pancreas/X_pca.npy",
        "V": "./data/real_data_benchmark/pancreas/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/pancreas/color_clusters.npy",
    },
    "dentate_gyrus": {
        "X": "./data/real_data_benchmark/dentate_gyrus/X_pca.npy",
        "V": "./data/real_data_benchmark/dentate_gyrus/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/dentate_gyrus/color_clusters.npy",
    },
    "larry": {
        "X": "./data/real_data_benchmark/larry/X_raw.npy",
        "V": "./data/real_data_benchmark/larry/V_raw.npy",
        "color": "./data/real_data_benchmark/larry/color_state.npy",
    },
}

In [ ]:
# -------------------------------------------------
# CHOOSE DATASET
# -------------------------------------------------
dataset_name = "pancreas"     # <----- CHANGE HERE
paths = DATASETS[dataset_name]


# -------------------------------------------------
# LOAD MATRICES
# -------------------------------------------------
X = np.load(paths["X"])
V = np.load(paths["V"])
C_raw = np.load(paths["color"])
C = make_dummy_continuous(C_raw)   # match FlowMap/Dynamo color scheme

proj = X + V
cell_names = [f"cell_{i}" for i in range(X.shape[0])]


# -------------------------------------------------
# SEND INPUT MATRICES TO R (modern rpy2 >= 3.5 API)
# -------------------------------------------------
with localconverter(default_converter + numpy2ri.converter + pandas2ri.converter):
    globalenv["curr"] = X.T
    globalenv["proj"] = proj.T
    globalenv["cell_names"] = StrVector(cell_names)


# -------------------------------------------------
# RUN VELOVIZ (R)
# -------------------------------------------------
r('''
    suppressPackageStartupMessages(library(veloviz))

    colnames(curr) <- cell_names
    colnames(proj) <- cell_names

    vv <- buildVeloviz(
            curr = curr,
            proj = proj,
            normalize.depth = FALSE, use.ods.genes = FALSE,
            alpha = 1, pca = FALSE, center = FALSE, scale = FALSE,
            k = 5, similarity.threshold = 0.25,
            distance.weight = 1, distance.threshold = 0.5,
            weighted = FALSE, verbose = FALSE
         )

    veloviz_embedding <- vv$fdg_coords
    cell_names_used   <- rownames(vv$fdg_coords)
''')


# -------------------------------------------------
# PULL RESULTS FROM R BACK INTO PYTHON (modern rpy2)
# -------------------------------------------------
with localconverter(default_converter + numpy2ri.converter + pandas2ri.converter):
    emb = np.array(r["veloviz_embedding"])
    cell_names_used = list(r["cell_names_used"])

keep = [int(s.split("_")[-1]) for s in cell_names_used]

Xk = X[keep]
Vk = V[keep]
Ck = C[keep]


# -------------------------------------------------
# PROJECT VELOCITY ONTO VELOVIZ BASIS
# -------------------------------------------------
adata = anndata.AnnData(Xk)
adata.layers["position"] = Xk
adata.layers["velocity"] = Vk
adata.obsm["X_veloviz"] = emb

scv.pp.neighbors(adata, use_rep="X")
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
scv.tl.velocity_embedding(adata, basis="veloviz")

V_emb = adata.obsm["velocity_veloviz"]


# -------------------------------------------------
# PLOT (consistent with FlowMap & Dynamo)
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    emb[:, 0], emb[:, 1],
    c=Ck, cmap="viridis",
    s=150, alpha=0.10, edgecolors="none"
)

ax.quiver(
    emb[:, 0], emb[:, 1],
    V_emb[:, 0], V_emb[:, 1],
    color="black", angles="xy", scale_units="xy",
    scale=0.5, width=0.004,
    headwidth=3, headlength=4, headaxislength=3
)

ax.set_title(f"Veloviz — {dataset_name}")
ax.set_aspect("equal")
ax.axis("off")
plt.show()


# -------------------------------------------------
# STORE RESULTS
# -------------------------------------------------
result = {
    "X": Xk,
    "V": Vk,
    "color": Ck,
    "embedding": emb,
    "velocity_emb": V_emb,
}

print("Veloviz embedding + velocity stored in `result`.")